# TCC · EDA de Marketing Mix Modeling com Google Meridian

**Dados simulados oficiais, foco GEO × TEMPO.** Não há estimação posterior, ROI ou recomendação de orçamento.

Execute de cima para baixo em runtime novo Python 3.12+ (CPU). A primeira execução instala dependências; isso pode levar alguns minutos. Se o Colab já tinha versões diferentes de NumPy/TensorFlow carregadas, reinicie a sessão após instalar e execute novamente.

O bootstrap clona o repositório público quando necessário, preserva instalações compatíveis e salva tudo na pasta do projeto. A fonte de dados e a API estão fixadas por commit/checksum. Para uma cópia histórica exata do projeto, configure `REPO_REF` com o commit da entrega. Os dados não possuem nomes de canais reais ou localização geográfica real.


In [1]:
# Execute esta célula primeiro. Em Colab novo, usa somente CPU.
import os, sys, subprocess, importlib.util, importlib.metadata
from pathlib import Path

REPO_URL = "https://github.com/thiagossilva17/MMM_Google_Meridian_Project.git"
REPO_REF = "main"  # Para repetir uma execução publicada, substitua pelo SHA do commit.
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
cwd = Path.cwd()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src/config.py").exists()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = cwd / "MMM_Google_Meridian_Project"
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", REPO_REF], check=True)
    elif not (PROJECT_ROOT / "src/config.py").exists():
        raise RuntimeError("Diretório de destino existe, mas não contém o projeto esperado.")

required = ["meridian", "pandas", "numpy", "scipy", "matplotlib", "statsmodels", "tabulate"]
missing = any(importlib.util.find_spec(package) is None for package in required)
try:
    wrong_meridian = importlib.metadata.version("google-meridian") != "2.1.0"
except importlib.metadata.PackageNotFoundError:
    wrong_meridian = True
if missing or wrong_meridian:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_ROOT / "requirements-colab.txt")], check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "outputs/.mplcache"))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
from src.config import SEED, ROOT
from src.data import load_panel
from src.reporting import show
import numpy as np
np.random.seed(SEED)
geo_time_panel = load_panel()
print(f"Colab: {IN_COLAB}; projeto: {ROOT}; painel: {geo_time_panel.shape}")


Colab: False; projeto: /workspace/scratch/5d823022ed95/tcc-meridian; painel: (6240, 19)


## 03 — As diferenças entre geos vão além do tamanho populacional?

### Pergunta
As diferenças entre geos vão além do tamanho populacional?

### Motivação
Um painel pode ter muitas linhas e ainda ser formado por cópias escaladas da série nacional. Precisamos separar tamanho, diferenças persistentes e diferenças de dinâmica.

### Método
Rankings e concentração do KPI, população, KPI semanal por 1.000 habitantes, sincronização, small multiples selecionados por critério. Decompor variância total = between + within; calcular R²_geo, R²_time e resíduo two-way em escala bruta e per capita.

### Interpretação antes de olhar os resultados
R²_geo alto indica diferenças persistentes. R²_time alto indica movimento comum. Within inclui movimento nacional, logo não equivale à informação adicional geo-temporal. No painel balanceado: R²_geo + R²_time + fração residual = 1. Essa identidade não demonstra causalidade.

**Dependências:** painel validado e funções em `src/geo_analysis.py`. O código reutilizável está comentado e pode ser aberto para inspecionar as fórmulas. Todos os resultados são calculados na execução, não preenchidos manualmente.


In [2]:
from src.geo_analysis import geo_analysis
report_03 = geo_analysis(geo_time_panel)
show(report_03)

### Resultado observado e conclusão parcial

O maior geo responde por 5.0% do KPI acumulado; os cinco maiores por 22.9%. A correlação populacional com KPI total é 0.978 (Spearman). A escala populacional precisa ser considerada antes de interpretar rankings.

O KPI tem 67.9% da variância between-geo e 32.1% within-geo. A identidade usa momentos populacionais (ddof=0), com peso igual por observação. R²_geo equivale à parcela between; R²_time quantifica semanas comuns. No painel balanceado, a parcela residual two-way do KPI é 30.7%; por habitante é 86.1%.

Within não significa automaticamente sinal geográfico novo: inclui choques temporais nacionais. Por isso apresentamos também o resíduo após remover simultaneamente geo e tempo. Essa decomposição é descritiva e depende da escala. Os small multiples usam geos escolhidos por critérios registrados; os cálculos e heatmaps incluem todos os geos. Geo0…Geo39 são rótulos sintéticos, sem fronteiras para um mapa geográfico real.

### O que ainda NÃO podemos concluir

As relações são descritivas. Não estimamos ROI, contribuição incremental ou efeitos causais.
Os dados são simulados: padrões não descrevem um mercado real e não validam, por si, a identificação de um MMM.

### CHECKPOINT

- **O que descobrimos?** As evidências numéricas acima e as tabelas exportadas respondem à pergunta deste capítulo.
- **Quais problemas apareceram?** Os limites e alertas estão descritos nos resultados; nenhuma observação foi excluída ou imputada.
- **O que falta investigar?** Mix, alocação territorial, concentração e variação de cada canal dentro dos geos.
- **Podemos avançar?** Podemos prosseguir com a investigação exploratória após revisar estes achados; isso não aprova automaticamente a modelagem.


**geo_summary** — 40 linhas; CSV completo em `outputs/tables/`.

,kpi_total,mean,std,min,max,population,kpi_share,population_share,kpi_per_1000_per_week
geo,,,,,,,,,
Geo36,3.290565e+09,2.109336e+07,5.507861e+06,4166830.8,36644916.0,969886.50,0.049909,0.044702,21748.281403
Geo39,3.222847e+09,2.065927e+07,4.438007e+06,6550282.0,30060602.0,798939.10,0.048882,0.036823,25858.383877
Geo10,3.093954e+09,1.983304e+07,6.371937e+06,5597731.5,33824700.0,994048.94,0.046927,0.045816,19951.772822
Geo23,2.836450e+09,1.818237e+07,5.226278e+06,5906972.0,36712692.0,981570.90,0.043022,0.045240,18523.746607
Geo31,2.684670e+09,1.720942e+07,4.773290e+06,5595800.0,31523720.0,873369.70,0.040720,0.040253,19704.623992
Geo7,2.578130e+09,1.652648e+07,4.180518e+06,5721620.0,28238984.0,733989.56,0.039104,0.033829,22515.954581
Geo2,2.468820e+09,1.582577e+07,4.684044e+06,4341767.0,29242514.0,892369.50,0.037446,0.041129,17734.548771
Geo25,2.437115e+09,1.562253e+07,4.305033e+06,4081507.0,25940226.0,779181.90,0.036965,0.035912,20049.916609
Geo17,2.201843e+09,1.411438e+07,4.340580e+06,3393230.5,25439212.0,777539.94,0.033396,0.035837,18152.606923


**decomposition** — 15 linhas; CSV completo em `outputs/tables/`.

,total_variance,between_geo,within_geo,between_share,within_share,r2_geo,r2_time,r2_geo_time,residual_geo_time_share
variable,,,,,,,,,
conversions,3.695331e+13,2.508695e+13,1.186637e+13,0.678882,0.321118,0.678882,0.014340,0.693223,0.306777
Channel0_impression,6.712189e+11,1.667112e+11,5.045076e+11,0.248371,0.751629,0.248371,0.136330,0.384701,0.615299
Channel1_impression,3.851114e+11,5.549878e+10,3.296126e+11,0.144111,0.855889,0.144111,0.167931,0.312042,0.687958
Channel2_impression,3.081801e+11,1.648447e+10,2.916956e+11,0.053490,0.946510,0.053490,0.167090,0.220580,0.779420
Channel3_impression,1.702277e+12,6.636814e+11,1.038596e+12,0.389879,0.610121,0.389879,0.181100,0.570978,0.429022
Channel4_impression,8.355853e+11,1.982119e+11,6.373734e+11,0.237213,0.762787,0.237213,0.190747,0.427960,0.572040
Channel0_spend,3.609065e+07,8.963869e+06,2.712678e+07,0.248371,0.751629,0.248371,0.136330,0.384701,0.615299
Channel1_spend,3.579739e+07,5.158797e+06,3.063859e+07,0.144111,0.855889,0.144111,0.167931,0.312042,0.687958
Channel2_spend,1.701728e+07,9.102497e+05,1.610703e+07,0.053490,0.946510,0.053490,0.167090,0.220580,0.779420


...

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]